# UST Micro Relative Value Analysis via CashSpline

This notebook demonstrates systematic Treasury relative value analysis using the `CashSpline` framework.

**Sections:**
1. Setup & Data Fetching
2. Cross-Sectional Spline Fit & Visualization
3. Per-CUSIP Yield Errors (Rich/Cheap)
4. Time Series of Spline Spreads (via TimeseriesBuilder)
5. Time Series of RMSE & Maturity Bucket RMSE
6. Multi-Method Comparison

In [1]:
%load_ext autoreload
%autoreload 2

import nest_asyncio
nest_asyncio.apply()

import warnings
warnings.filterwarnings("ignore")

import datetime
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"

import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-dark")

In [2]:
import sys
sys.path.append("../../")

from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP
from MDP.FixedRateBonds.cash_spline import (
    CashSpline,
    CashSplineBuilder,
    CashSplineConfig,
    JPM_PAR_CURVE_CONFIG,
    MMSS_SPLINE_CONFIG,
    SplineMethod,
)
from Query.FixedRateBonds.FixedRateBondQuery import FixedRateBondQuery
from Query.FixedRateBonds.FixedRateBondValue import FixedRateBondValue
from Query.FixedRateBonds.spline_values import MATURITY_BUCKETS
from TB.FixedRateBondsTB import FixedRateBondsTB
from TB.TimeseriesBuilder import TimeseriesBuilder
from RVUtils.ust_viz import plot_usts

## 1. Setup & Data Fetching

In [3]:
as_of = datetime.date(2026, 5, 27)

usts_mdp = FixedRateBondsMDP(source="USTS_FEDINVEST_WSJ_LIVE-QL")
frb_tb = FixedRateBondsTB(usts_mdp, show_tqdm=True)
ts_builder = TimeseriesBuilder(fixedratebonds_tb=frb_tb)

In [5]:
# Build spline using the MDP integration
with usts_mdp:
    spline = usts_mdp.fetch_cash_spline(as_of)

print(f"Date:      {spline.as_of_date}")
print(f"Bonds:     {len(spline.fit_ttm)}")
print(f"Method:    {spline.config.method}")
print(f"RMSE:      {spline.rmse:.2f} bp")
print(f"MAE:       {spline.mae:.2f} bp")
print(f"TTM range: [{spline.fit_ttm.min():.1f}, {spline.fit_ttm.max():.1f}] years")

Date:      2026-05-27
Bonds:     273
Method:    b_spline_with_knots
RMSE:      1.42 bp
MAE:       0.87 bp
TTM range: [1.0, 29.2] years


## 2. Cross-Sectional Spline Fit & Visualization

In [9]:
# Build the fitted curve DataFrame for visualization
summary = spline.to_frame().reset_index()

# Smooth spline evaluation for plotting
ttm_grid = np.linspace(max(0.5, spline.fit_ttm.min()), spline.fit_ttm.max(), 500)
ytm_grid = spline.yield_at(ttm_grid)

# --- Interactive Plotly scatter + spline curve ---
fig = go.Figure()

# Scatter: observed bond yields
fig.add_trace(
    go.Scatter(
        x=summary["ttm"],
        y=summary["observed"],
        mode="markers",
        marker=dict(size=6, color=summary["yield_error_bp"], colorscale="RdYlGn_r", colorbar=dict(title="Yield Error (bp)"), cmin=-3, cmax=3, showscale=False),
        text=summary["cusip"],
        hovertemplate="<b>%{text}</b><br>TTM: %{x:.1f}Y<br>YTM: %{y:.3f}%<br>Error: %{marker.color:.2f}bp<extra></extra>",
        name="Bonds",
    )
)

# Line: fitted spline
fig.add_trace(
    go.Scatter(
        x=ttm_grid,
        y=ytm_grid,
        mode="lines",
        line=dict(color="red", width=2),
        name=f"Fitted Spline ({spline.config.method})",
    )
)

fig.update_layout(
    template="plotly_dark",
    title=f"UST Par Curve — {as_of} — RMSE: {spline.rmse:.2f}bp ({len(spline.fit_ttm)} bonds)",
    xaxis_title="Time to Maturity (years)",
    yaxis_title="Yield to Maturity (%)",
    height=700,
    hovermode="closest",
)
fig.show()

## 3. Per-CUSIP Yield Errors (Rich / Cheap Analysis)

In [10]:
# Yield error bar chart — sorted by TTM
errs = spline.to_frame().reset_index().sort_values("ttm")
errs["color"] = errs["yield_error_bp"].apply(lambda x: "cheap" if x > 0 else "rich")

fig_err = go.Figure()
fig_err.add_trace(go.Bar(
    x=errs["ttm"], y=errs["yield_error_bp"],
    marker_color=errs["yield_error_bp"].apply(lambda x: "#2ca02c" if x > 0 else "#d62728"),
    text=errs["cusip"],
    hovertemplate="<b>%{text}</b><br>TTM: %{x:.1f}Y<br>Error: %{y:.2f}bp<extra></extra>",
    name="Yield Error",
))

fig_err.add_hline(y=0, line_dash="dash", line_color="white", opacity=0.5)

fig_err.update_layout(
    template="plotly_dark",
    title=f"Bond-Level Yield Errors vs Spline — {as_of}",
    xaxis_title="Time to Maturity (years)",
    yaxis_title="Yield Error (bp) — Green=Cheap, Red=Rich",
    height=500,
)
fig_err.show()

In [11]:
# Top richest and cheapest bonds
n_show = 10
ye = spline.yield_errors.sort_values()
print(f"=== Top {n_show} RICHEST bonds (most negative yield error) ===")
print(ye.head(n_show).to_string())
print(f"\n=== Top {n_show} CHEAPEST bonds (most positive yield error) ===")
print(ye.tail(n_show).to_string())

=== Top 10 RICHEST bonds (most negative yield error) ===
912810FG8   -10.549241
912810QN1    -5.105729
912810QQ4    -4.724942
912810FT0    -4.657234
912810QS0    -3.980993
912810UL0    -3.978740
912810QL5    -3.135877
912810UJ5    -2.934544
912810PW2    -2.854269
91282CAL5    -2.355994

=== Top 10 CHEAPEST bonds (most positive yield error) ===
912810RJ9    2.266695
912810SW9    2.394271
912810RB6    2.410371
91282CNE7    2.417376
91282CNC1    2.738643
91282CNT4    2.927045
912810RK6    2.965456
912810ST6    4.464123
912810SQ2    5.962580
912810SR0    6.434458


In [12]:
# Maturity-bucket RMSE breakdown (JPM style)
bucket_data = []
for name, (lo, hi) in MATURITY_BUCKETS.items():
    rmse = spline.rmse_bucket(lo, hi)
    mask = (spline.fit_ttm >= lo) & (spline.fit_ttm < hi)
    n_bonds = int(mask.sum())
    bucket_data.append({"bucket": name, "rmse_bp": rmse, "n_bonds": n_bonds})

bucket_df = pd.DataFrame(bucket_data)
print(f"\nMaturity Bucket RMSE — {as_of}")
print(bucket_df.to_string(index=False))

fig_bkt = go.Figure(go.Bar(
    x=bucket_df["bucket"], y=bucket_df["rmse_bp"],
    text=bucket_df["n_bonds"].apply(lambda x: f"n={x}"),
    textposition="outside",
    marker_color="#636efa",
))
fig_bkt.update_layout(
    template="plotly_dark",
    title=f"RMSE by Maturity Bucket — {as_of} — Overall RMSE: {spline.rmse:.2f}bp",
    xaxis_title="Maturity Bucket",
    yaxis_title="RMSE (bp)",
    height=450,
)
fig_bkt.show()


Maturity Bucket RMSE — 2026-05-27
bucket  rmse_bp  n_bonds
  0-2Y 0.986071       50
  2-3Y 1.752275       40
  3-5Y 0.571779       54
  5-7Y 0.545813       29
 7-10Y 2.155685       10
10-15Y 3.275851       17
15-20Y 1.741784       36
20-30Y 0.642381       37


In [24]:
# Define date range and CUSIPs of interest
ts_start = datetime.date(2025, 1, 2)
ts_end = datetime.date(2026, 5, 27)

cusips_of_interest = ["CT10", "CT5", "CT2", "CT30", "CT7"]

## 5. Time Series of RMSE & Maturity Bucket RMSE

Track the overall spline fit quality over time — replicating JPM's RMSE monitoring.

In [ ]:
# Build RMSE + bucket RMSE queries
rmse_queries = [
    # Overall RMSE
    FixedRateBondQuery(
        cusip="912810UA4",
        value=FixedRateBondValue.SPLINE_SPREAD,
    ),
    # # Key bucket RMSEs
    # FixedRateBondQuery(
    #     cusip="CT10",
    #     value=FixedRateBondValue.SPLINE_RMSE_BUCKET,
    #     value_kwargs={"bucket": "2-3Y"},
    # ),
    # FixedRateBondQuery(
    #     cusip="CT10",
    #     value=FixedRateBondValue.SPLINE_RMSE_BUCKET,
    #     value_kwargs={"bucket": "7-10Y"},
    # ),
    # FixedRateBondQuery(
    #     cusip="CT10",
    #     value=FixedRateBondValue.SPLINE_RMSE_BUCKET,
    #     value_kwargs={"bucket": "20-30Y"},
    # ),
]

print(f"Fetching RMSE time series from {ts_start} to {ts_end}...")
df_rmse = ts_builder.get_timeseries(
    start=ts_start,
    end=ts_end,
    queries=rmse_queries,
    n_jobs=4,
    drop_multilevel_cols=True,
)
print(f"Shape: {df_rmse.shape}")
df_rmse.tail()

Fetching RMSE time series from 2025-01-02 to 2026-05-27...


FETCHING PRICERS:   0%|          | 0/2 [00:00<?, ?it/s]

PRICING FIXED-RATE BONDS.:   0%|          | 0/42 [00:00<?, ?it/s]WARNING	Thread(ThreadPoolExecutor-10_2) Query.FixedRateBonds.spline_values:spline_values.py:compute_spline_for_date()- Spline fit failed for 2026-04-01: Insufficient data after filtering: 1 bonds (need >= 4)
WARNING	Thread(ThreadPoolExecutor-10_0) Query.FixedRateBonds.spline_values:spline_values.py:compute_spline_for_date()- Spline fit failed for 2026-03-30: Insufficient data after filtering: 1 bonds (need >= 4)
WARNING	Thread(ThreadPoolExecutor-10_1) Query.FixedRateBonds.spline_values:spline_values.py:compute_spline_for_date()- Spline fit failed for 2026-03-31: Insufficient data after filtering: 1 bonds (need >= 4)
WARNING	Thread(ThreadPoolExecutor-10_3) Query.FixedRateBonds.spline_values:spline_values.py:compute_spline_for_date()- Spline fit failed for 2026-04-02: Insufficient data after filtering: 1 bonds (need >= 4)
PRICING FIXED-RATE BONDS.:  31%|███       | 13

Shape: (350, 1)


,912810UA4 OUTRIGHT SPLINE_SPREAD
Date,
2026-05-20,-0.302558
2026-05-21,-1.436641
2026-05-22,NaN
2026-05-26,-0.306715
2026-05-27,-0.273400


: 

In [23]:
# Plot RMSE time series
fig_rmse = go.Figure()

# Clean up column names for display
rename_map = {}
for col in df_rmse.columns:
    if "SPLINE_RMSE_BUCKET" in col:
        rename_map[col] = col.split("SPLINE_RMSE_BUCKET")[0].strip().split()[-1] if "bucket" in col.lower() else col
    elif "SPLINE_RMSE" in col:
        rename_map[col] = "Overall RMSE"

for col in df_rmse.columns:
    display_name = rename_map.get(col, col)
    is_overall = "RMSE" in col and "BUCKET" not in col
    fig_rmse.add_trace(go.Scatter(
        x=df_rmse.index, y=df_rmse[col],
        mode="lines",
        name=display_name,
        line=dict(width=3 if is_overall else 1.5),
    ))

fig_rmse.update_layout(
    template="plotly_dark",
    title=f"Spline RMSE Time Series — {ts_start} to {ts_end}",
    xaxis_title="Date",
    yaxis_title="RMSE (bp)",
    height=550,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig_rmse.show()

## 6. Multi-Method Comparison

Compare different spline fitting methods on the same date to understand model sensitivity.

In [ ]:
# Compare multiple fitting methods on the same data
# Note: some methods (PCHIP, Smoothing) require unique TTMs, so we deduplicate
dedup_df = pd.DataFrame({"ttm": spline.fit_ttm, "y": spline.fit_y})
dedup_df = dedup_df.groupby("ttm", as_index=False)["y"].mean().sort_values("ttm")
dedup_ttm = dedup_df["ttm"].to_numpy()
dedup_y = dedup_df["y"].to_numpy()

methods_to_compare = [
    ("B-Spline (JPM knots)", JPM_PAR_CURVE_CONFIG),
    ("PCHIP", CashSplineConfig(method="pchip", exclude_ranks=(), min_ttm=0.0, min_points=3)),
    ("Smoothing Spline", CashSplineConfig(method="smoothing_spline", exclude_ranks=(), min_ttm=0.0, min_points=3)),
    ("LOESS (frac=0.3)", CashSplineConfig(method="loess", loess_frac=0.3, exclude_ranks=(), min_ttm=0.0, min_points=3)),
    ("Nelson-Siegel", CashSplineConfig(method="nelson_siegel", exclude_ranks=(), min_ttm=0.0, min_points=3)),
]

fig_cmp = go.Figure()

# Plot bonds (from the first spline)
fig_cmp.add_trace(go.Scatter(
    x=spline.fit_ttm, y=spline.fit_y,
    mode="markers", marker=dict(size=4, color="white", opacity=0.5),
    name="Bonds", showlegend=True,
))

colors = px.colors.qualitative.Plotly
for i, (label, cfg) in enumerate(methods_to_compare):
    builder = CashSplineBuilder(cfg)
    try:
        s = builder.fit(ttm=dedup_ttm, y=dedup_y)
        y_eval = s.yield_at(ttm_grid)
        fig_cmp.add_trace(go.Scatter(
            x=ttm_grid, y=y_eval,
            mode="lines", name=f"{label} (RMSE={s.rmse:.2f}bp)",
            line=dict(color=colors[i % len(colors)], width=2),
        ))
    except Exception as e:
        print(f"{label}: FAILED — {e}")

fig_cmp.update_layout(
    template="plotly_dark",
    title=f"Spline Method Comparison — {as_of}",
    xaxis_title="Time to Maturity (years)",
    yaxis_title="Yield to Maturity (%)",
    height=700,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig_cmp.show()